# FHIR API validation

Validate HAPI FHIR R4 connectivity, metadata and recent incremental search behaviour.

In [ ]:
import requests
from datetime import datetime, timedelta, timezone

BASE_URL = "https://hapi.fhir.org/baseR4"
RESOURCES = ["Patient", "Encounter", "Observation", "Condition"]
session = requests.Session()
session.headers.update({"Accept": "application/fhir+json"})

metadata_response = session.get(f"{BASE_URL}/metadata", timeout=60)
metadata_response.raise_for_status()
capability = metadata_response.json()
print("FHIR version:", capability.get("fhirVersion"))

since = (datetime.now(timezone.utc) - timedelta(hours=72)).strftime("%Y-%m-%dT%H:%M:%SZ")
for resource in RESOURCES:
    response = session.get(f"{BASE_URL}/{resource}", params={"_lastUpdated": f"ge{since}", "_count": 5}, timeout=60)
    response.raise_for_status()
    bundle = response.json()
    print(resource, "total:", bundle.get("total"), "entries:", len(bundle.get("entry", [])))
    print("next link:", next((x.get("url") for x in bundle.get("link", []) if x.get("relation") == "next"), None))